# CodeTuneEfficiency on a free GPU

Runs the full benchmark on **Google Colab (free T4)** or **Kaggle (free weekly GPU hours)**.
Nothing here costs money and no paid tier is required.

| | Colab free | Kaggle free |
|---|---|---|
| GPU | T4, 16 GB | T4 x2 or P100, 16 GB |
| Session limit | ~12 h | 12 h, 30 GPU-h/week |
| Enable it | Runtime → Change runtime type → T4 GPU | Settings → Accelerator → GPU |

With 16 GB instead of the 6 GB this was developed on you can raise `batch_size` to 32
and drop `grad_accum` to 1, which is roughly 3x faster.

**Download `results/` before the session ends — Colab and Kaggle both wipe the disk on disconnect.**

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
# Colab and Kaggle already ship a CUDA build of torch, so only the rest is installed.
#
# NOTE: the work lives on a branch. Cloning the default branch gets you the
# upstream artifact with no codetune/ package, and every cell below fails.
BRANCH = "feat/peft-code-benchmark"

!git clone --branch {BRANCH} --single-branch https://github.com/Ssavan99/CodeTuneEfficiency.git
%cd CodeTuneEfficiency
!pip install -q transformers==4.44.2 datasets==2.21.0 peft==0.12.0 accelerate==0.34.2 pyyaml==6.0.2

import pathlib
assert pathlib.Path("codetune/train.py").exists(), (
    f"codetune/ is missing - is branch {BRANCH} pushed to GitHub?"
)
print("repo ready on", BRANCH)

Cloning into 'CodeTuneEfficiency'...
remote: Enumerating objects: 301, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 301 (delta 92), reused 125 (delta 49), pack-reused 95 (from 1)
Receiving objects: 100% (301/301), 1.48 MiB | 4.22 MiB/s, done.
Resolving deltas: 100% (131/131), done.
/content/CodeTuneEfficiency
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566

In [3]:
# Both datasets come from the public CodeXGLUE copies on the Hub - free, and
# no Hugging Face account or token is needed. BigCloneBench is ~900k pairs,
# so this cell takes a few minutes the first time.
!python -m codetune prepare

[prepare] clone: downloading BigCloneBench from the Hugging Face Hub
Generating train split: 100% 901028/901028 [00:14<00:00, 63015.29 examples/s]
Generating validation split: 100% 415416/415416 [00:08<00:00, 47416.39 examples/s]
Generating test split: 100% 415416/415416 [00:04<00:00, 83822.64 examples/s] 
[prepare] clone/train.txt (901,028 pairs)
[prepare] clone/valid.txt (415,416 pairs)
[prepare] clone/test.txt (415,416 pairs)
[prepare] clone/data.jsonl (9,126 unique functions)
[prepare] /content/CodeTuneEfficiency-model/defect.zip not found; downloading Devign from the Hugging Face Hub
Generating train split: 100% 21854/21854 [00:00<00:00, 148648.27 examples/s]
Generating validation split: 100% 2732/2732 [00:00<00:00, 134013.67 examples/s]
Generating test split: 100% 2732/2732 [00:00<00:00, 146448.19 examples/s]
[prepare] defect/train.jsonl (21854 rows)
[prepare] defect/valid.jsonl (2732 rows)
[prepare] defect/test.jsonl (2732 rows)


In [4]:
# Sanity check first - a couple of minutes, proves the pipeline works.
!python -m codetune run --config configs/smoke.yaml

2026-08-10 17:52:23.387291: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 199kB/s]
config.json: 100% 498/498 [00:00<00:00, 4.60MB/s]
vocab.json: 899kB [00:00, 32.5MB/s]
merges.txt: 456kB [00:00, 97.1MB/s]
special_tokens_map.json: 100% 150/150 [00:00<00:00, 1.63MB/s]
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
pytorch_model.bin: 100% 499M/499M [00:06<00:0

In [5]:
# A 16 GB T4 fits a far larger batch than the 6 GB card this was written on,
# and unlike a GTX 1660 Ti it has tensor cores, so fp16 is a real speedup.
# Scale is also raised now that the compute budget allows it.
import yaml

for name, limit in (("configs/defect.yaml", None), ("configs/clone.yaml", 20000)):
    cfg = yaml.safe_load(open(name))
    cfg["batch_size"], cfg["grad_accum"] = 32, 1
    cfg["max_length"] = 256
    if limit is None:
        cfg.pop("limit_train", None)   # full 21,854-example Devign train set
    else:
        cfg["limit_train"] = limit
    yaml.safe_dump(cfg, open(name, "w"), sort_keys=False)
    print(name, "->", cfg)

configs/defect.yaml -> {'task': 'defect', 'epochs': 2, 'lr': 5e-05, 'peft_lr': 0.0001, 'batch_size': 32, 'grad_accum': 1, 'max_length': 256, 'weight_decay': 0.01, 'warmup_ratio': 0.06, 'fp16': True, 'limit_eval': 1000, 'output_dir': 'results'}
configs/clone.yaml -> {'task': 'clone', 'epochs': 2, 'lr': 5e-05, 'peft_lr': 0.0001, 'batch_size': 32, 'grad_accum': 1, 'max_length': 256, 'weight_decay': 0.01, 'warmup_ratio': 0.06, 'fp16': True, 'limit_train': 20000, 'limit_eval': 1000, 'output_dir': 'results'}


In [6]:
# 4 methods x 3 seeds. `grid` skips runs that already have a result file, so
# re-running after a disconnect resumes instead of starting over.
!python -m codetune grid --config configs/defect.yaml

2026-08-10 17:53:55.481866: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN thi

In [7]:
!python -m codetune grid --config configs/clone.yaml

2026-08-10 20:07:55.504301: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN thi

In [8]:
!python -m codetune aggregate
!python -m codetune plot

[aggregate] 24 runs -> results/summary.csv, results/summary.md
[aggregate] refreshed the results section of README.md
### clone — 20,000 train / 1,000 test, 2 epochs

| Method | Seeds | Accuracy | Macro F1 | Positive F1 | Trainable | Delta ckpt | Peak VRAM | Train time |
|---|---|---|---|---|---|---|---|---|
| `full` | 3 | 93.37 ± 0.91 | 86.77 ± 1.76 | 77.42 ± 2.99 | 100.000% (124,647,170) | 475.49 MB | 7,374 MB | 11.2 min |
| `bitfit` | 3 | 89.20 ± 0.62 | 80.25 ± 0.61 | 66.95 ± 0.85 | 0.560% (694,274) | 2.65 MB | 7,502 MB | 8.8 min |
| `lora` | 3 | 91.73 ± 0.81 | 84.06 ± 1.39 | 73.00 ± 2.31 | 0.710% (887,042) | 3.38 MB | 7,508 MB | 9.3 min |
| `parallel_adapter` | 3 | 92.27 ± 0.64 | 84.96 ± 1.39 | 74.48 ± 2.42 | 0.720% (896,450) | 3.42 MB | 7,508 MB | 8.5 min |

### defect — 21,854 train / 1,000 test, 2 epochs

| Method | Seeds | Accuracy | Macro F1 | Positive F1 | Trainable | Delta ckpt | Peak VRAM | Train time |
|---|---|---|---|---|---|---|---|---|
| `full` | 3 | 64.33 ± 1.37 | 62.

In [9]:
# Save the results off the ephemeral disk before the session ends.
!zip -qr results.zip results && echo "wrote results.zip"
try:
    from google.colab import files

    files.download("results.zip")
except ImportError:
    print("On Kaggle: results.zip is in the working directory - use the Output panel to download it.")

wrote results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Bringing the results home

Unzip `results.zip` into the repo's `results/` directory on your machine, then run:

```bash
python -m codetune aggregate && python -m codetune plot
```

That fills the README's results tables, writes `results/summary.csv` and the figures.
Commit `results/` and the refreshed `README.md`.